# YEJOO 재고 수불부 STEP 2 
# (V18 기반 속도개선버전)

### -*- coding: utf-8 -*-  # 인코딩 설정

YEJOO 재고수불부 자동화 V17 (노트북 → 단일 py 파일)
- 전처리
- STEP0 / STEP0-1
- STEP00 (보세 → 본사 이동 날짜 당김)
- STEP1 ~ STEP5

In [ ]:
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 처리
import time  # 소요 시간 측정
from datetime import timedelta  # 날짜 처리
from datetime import datetime  # 중간저장 타임스탬프
from tqdm.auto import tqdm  # type: ignore

# =========================  # 구분선
# 0) 경로  # 섹션
# =========================  # 구분선
PATH_SUFUL = r"../input data/01조정_수불부.xlsx"  # 수불부 경로
PATH_SUBMIT = r"../input data/01조정_제출본.xlsx"  # 제출본 경로(미사용)
PATH_BONDED = r"../input data/00전처리_보세수불부.xlsx"  # 보세수불부 경로
OUT_XLSX = r"입고이동_최종_ver2.xlsx"  # 출력 경로

# =========================  # 구분선
# 1) 컬럼명  # 섹션
# =========================  # 구분선
DATE_COL = "일자"  # 날짜
ITEM_COL = "품목코드"  # 품목코드
NAME_COL = "품목명"  # 품목명
PARTY_COL = "거래처명"  # 거래처명
WH_COL = "창고명"  # 창고명

IN_QTY_COL = "입고수량"  # 입고수량
OUT_QTY_COL = "출고수량"  # 출고수량
STOCK_COL = "재고수량"  # 재고수량

IN_PRICE_COL = "입고단가"  # 입고단가
IN_AMT_COL = "입고금액"  # 입고금액

# =========================  # 구분선
# 2) 옵션  # 섹션
# =========================  # 구분선
STEP00_MAX_PULL_DAYS = 60  # 탐색창(일)
STEP00_WORST_THRESHOLD = -1.0  # 음수 기준(이하만)
VERBOSE = True  # 로그 출력

BONDED_MOVE_TEXT = "[이동] 07보세창고 → 01본사창고"  # 보세→본사 텍스트

STEP00_SAVE_EVERY_N_SUCCESS = 0  # 0이면 중간저장 안 함 (마지막에만 최종 저장)
STEP00_SAVE_PREFIX = "중간저장_STEP00"  # 중간저장 접두어 (STEP00_SAVE_EVERY_N_SUCCESS > 0일 때만 사용)
STEP00_MID_SAVE_FULL = False  # 중간저장에 df_main/df_bonded까지 같이 저장할지(⚠️ True면 매우 느림/중단 위험)

EPS_NEG = -1  # ✅ -0.000... 같은 오차 방지

# =========================  # 구분선
# 3) 유틸  # 섹션
# =========================  # 구분선
def normalize_item_code(x):  # 품목코드 정리 (수불부 "3745" vs 보세 "3745.0" → "3745" 통일, 매칭용)
    if pd.isna(x):  # 결측이면
        return ""  # 빈값
    s = str(x).strip()  # 공백 제거
    try:  # 숫자로 읽히면(보세는 엑셀에서 float로 읽힘) 정수 문자열로 통일
        v = float(s)
        if v == int(v):  # 소수부 없음 (3745.0 등)
            return str(int(v))  # "3745"
        return s  # 3745.5 같은 실제 소수는 그대로
    except (ValueError, TypeError):  # 숫자 아님
        return s  # 문자열 그대로

def safe_date(x):  # 날짜 변환
    return pd.to_datetime(x, errors="coerce", format="mixed")  # 안전 변환

def to_float_series(s):  # float 변환
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype("float64")  # float 통일

def v_or_blank(v):  # 문자열 안전
    if pd.isna(v):  # 결측이면
        return ""  # 빈값
    return str(v)  # 문자열

def str_contains(hay, needle):  # 포함 검사
    if pd.isna(hay):  # 결측이면
        return False  # 아니야
    return needle in str(hay)  # 포함

def same_month(a, b):  # 같은 월인지
    a = pd.Timestamp(a)  # 변환
    b = pd.Timestamp(b)  # 변환
    return (a.year == b.year) and (a.month == b.month)  # 연/월 비교

def qty_equal(a, b, tol=1e-9):  # 수량 일치
    a = float(a)  # float
    b = float(b)  # float
    return abs(a - b) <= tol  # 오차 허용

# =========================  # 구분선
# 4) 로딩 / 전처리  # 섹션
# =========================  # 구분선
def load_excel(path):  # 엑셀 로드
    return pd.read_excel(path)  # 읽기

def preprocess(df):  # 전처리
    df = df.copy()  # 복사
    df[DATE_COL] = safe_date(df[DATE_COL])  # 날짜 변환
    df = df.dropna(subset=[DATE_COL]).copy()  # 날짜 없는 행 제거

    df[ITEM_COL] = df[ITEM_COL].apply(normalize_item_code)  # 품목코드 정리

    if WH_COL not in df.columns:  # 창고명 없으면
        df[WH_COL] = ""  # 생성
    if PARTY_COL not in df.columns:  # 거래처명 없으면
        df[PARTY_COL] = ""  # 생성
    if NAME_COL not in df.columns:  # 품목명 없으면
        df[NAME_COL] = ""  # 생성
    if IN_PRICE_COL not in df.columns:  # 단가 없으면
        df[IN_PRICE_COL] = 0.0  # 생성
    if IN_AMT_COL not in df.columns:  # 금액 없으면
        df[IN_AMT_COL] = 0.0  # 생성

    df[IN_QTY_COL] = to_float_series(df.get(IN_QTY_COL, 0.0))  # 입고 float
    df[OUT_QTY_COL] = to_float_series(df.get(OUT_QTY_COL, 0.0))  # 출고 float
    df[STOCK_COL] = to_float_series(df.get(STOCK_COL, 0.0))  # 재고 float
    df[IN_PRICE_COL] = to_float_series(df.get(IN_PRICE_COL, 0.0))  # 단가 float
    df[IN_AMT_COL] = to_float_series(df.get(IN_AMT_COL, 0.0))  # 금액 float

    return df  # 반환

# =========================  # 구분선
# 5) 재고 재계산(창고+품목)  # 섹션 - 단일 그룹 재계산 + 전체 재계산
# =========================  # 구분선
def recalc_inventory_group(g):  # 단일 그룹만 재계산 (이동 후 해당 그룹만 갱신용)
    g = g.copy()
    if "_ord" not in g.columns:
        g["_ord"] = np.arange(len(g), dtype="int64")
    sort_cols = [DATE_COL]
    if "_moved_first" in g.columns:
        sort_cols.append("_moved_first")
    sort_cols.append("_ord")
    g = g.sort_values(sort_cols)
    in_arr = g[IN_QTY_COL].to_numpy(dtype=np.float64)
    out_arr = g[OUT_QTY_COL].to_numpy(dtype=np.float64)
    diff = in_arr - out_arr
    i0 = None
    for i in range(len(g)):
        v = g.iloc[i][STOCK_COL]
        if pd.notna(v) and str(v).strip() != "":
            i0 = i
            break
    if i0 is None:
        run_arr = np.cumsum(diff)
    else:
        base = float(g.iloc[i0][STOCK_COL])  # 기초재고(해당 행 기준)
        run_arr = np.empty(len(g), dtype=np.float64)
        run_arr[i0] = base
        # i0 이전 행: 기초에서 역산 (이전 행 재고 = 기초 - diff[i0] - ... - diff[i0-k])
        if i0 > 0:
            rev_diff = diff[i0:0:-1]  # i0, i0-1, ..., 1
            run_arr[0:i0] = base - np.cumsum(rev_diff)
        if i0 + 1 < len(g):
            run_arr[i0 + 1 :] = base + np.cumsum(diff[i0 + 1 :])
    g = g.copy()
    g[STOCK_COL] = run_arr
    return g

def recalc_inventory(df):  # 재고 재계산 (전체: 그룹별 recalc_inventory_group 호출)
    df = df.copy()
    df["_ord"] = np.arange(len(df), dtype="int64")
    sort_cols = [WH_COL, ITEM_COL, DATE_COL, "_moved_first", "_ord"] if "_moved_first" in df.columns else [WH_COL, ITEM_COL, DATE_COL, "_ord"]
    df = df.sort_values(sort_cols)
    out = [recalc_inventory_group(g) for (_, _), g in df.groupby([WH_COL, ITEM_COL], sort=False)]
    res = pd.concat(out, ignore_index=True) if len(out) > 0 else df.copy()
    if "_ord" in res.columns:
        res = res.drop(columns=["_ord"])
    return res

# =========================  # 구분선
# 6) 음수 run 찾기(연속 음수 구간)  # 섹션
# =========================  # 구분선
def find_all_negative_runs(df_item_sorted):  # 음수 run 목록 (numpy 벡터화)
    stocks = np.asarray(df_item_sorted[STOCK_COL], dtype=np.float64)
    dates = df_item_sorted[DATE_COL].to_numpy()
    is_neg = stocks < -EPS_NEG
    # run 시작: 비음수→음수 전환 위치 (맨 앞이 음수면 0 포함)
    pad = np.concatenate([[False], is_neg])
    starts = np.nonzero(pad[1:] & ~pad[:-1])[0]
    runs = []
    for si in starts:
        tail = is_neg[si:]
        pos = np.where(~tail)[0]
        ei = int(si + pos[0]) if len(pos) > 0 else len(stocks)
        worst = float(np.min(stocks[si:ei]))
        need_qty = abs(float(stocks[si]))
        runs.append({
            "run_start": pd.Timestamp(dates[si]),
            "start_idx": int(si),
            "end_idx": ei,
            "need_qty": need_qty,
            "worst": worst,
        })
    return runs

# =========================  # 구분선
# 7) STEP00  # 섹션
#  - ✅ 음수 run 있을 때만 이동  # 주석
#  - ✅ 이동 목표는 "해당 run_start"  # 주석
#  - ✅ 이력이력은 성공만 기록  # 주석
#  - ✅ 보세는 (날짜+품목+수량) 완전일치 후 같이 당김  # 주석
# =========================  # 구분선
def step00_pull_all_wh(df_main, df_bonded):  # STEP00 실행
    df_main = df_main.copy()  # 복사
    df_bonded = df_bonded.copy()  # 복사
    # 이동한 전표는 해당일 맨 앞에 오도록 정렬용 플래그 (0=맨앞, 1=그 외)
    df_main["_moved_first"] = 1
    df_bonded["_moved_first"] = 1

    history_rows = []  # ✅ 성공 이력만
    success_count = 0  # 성공 카운트
    used_main_idx = set()  # 본사 입고 재사용 금지 (한 번 이동한 전표는 다시 사용 안 함)
    used_bonded_idx = set()  # 보세에서 이미 이동한 전표(출고+구매) 재사용 금지
    failed_run_keys = set()  # 실패 run 스킵

    df_main = recalc_inventory(df_main)  # 재계산
    df_bonded = recalc_inventory(df_bonded)  # 재계산

    # 음수 재고 있는 (창고, 품목)만 처리 → 루프 수 대폭 감소
    min_stock = df_main.groupby([WH_COL, ITEM_COL])[STOCK_COL].min()
    keys = min_stock[min_stock < -EPS_NEG].index.tolist()

    iter_keys = tqdm(keys, desc="STEP00(본사)", total=len(keys)) if tqdm else keys  # 진행바
    for wh, code in iter_keys:  # 그룹 반복
        if tqdm:
            iter_keys.set_postfix_str(f"{wh} | {code} | success={success_count}")
        while True:  # run 반복
            g = df_main[(df_main[WH_COL] == wh) & (df_main[ITEM_COL] == code)].copy()  # 그룹 추출
            if len(g) == 0:  # 없으면
                break  # 종료

            # ✅ V18 방식: 이 루프에 들어오기 전/이전 성공 때마다 recalc_inventory로 이미 재계산되어 있다고 가정하고 사용
            # (필요하면 성능을 희생하고 항상 df_main 전체를 재계산하는 쪽이 정확도가 높음)

            # ✅ 음수 자체가 없으면 절대 이동 금지  # 주석
            if float(g[STOCK_COL].min()) >= -EPS_NEG:  # 음수 없음
                break  # 종료

            g = g.sort_values(DATE_COL).reset_index(drop=False)  # 정렬(+원본 index)
            runs = find_all_negative_runs(g)  # run 찾기
            if len(runs) == 0:  # run 없으면
                break  # 종료

            run = None  # 선택 run
            for r in runs:  # 앞 run부터
                rk = (wh, code, pd.Timestamp(r["run_start"]).date())  # 키
                if rk not in failed_run_keys:  # 실패 run 아니면
                    run = r  # 선택
                    break  # 중단
            if run is None:  # 다 실패면
                break  # 종료

            # ✅ 기준 이하 음수만 처리  # 주석
            if float(run["worst"]) > float(STEP00_WORST_THRESHOLD):  # 덜 심각하면
                break  # 종료

            run_start = pd.Timestamp(run["run_start"])  # run 시작일
            end_day = run_start + timedelta(days=int(STEP00_MAX_PULL_DAYS))  # 탐색 끝일

            base_mask = (  # 후보 조건
                (df_main[WH_COL] == wh) &  # 같은 창고
                (df_main[ITEM_COL] == code) &  # 같은 품목
                (df_main[DATE_COL] > run_start) &  # run 이후
                (df_main[DATE_COL] <= end_day) &  # 창 안
                (df_main[IN_QTY_COL] > 0.0)  # 입고만
            )  # 마스크

            cand = df_main[base_mask].copy()  # 후보
            if len(cand) > 0:
                cand = cand[~cand.index.isin(list(used_main_idx))].copy()  # 이미 이동한 전표 제외

            fail_reason = ""  # 실패 사유
            bonded_fail = False  # 보세 실패
            move_type = ""  # 타입
            pick_idx = None  # 선택 idx

            bonded_new_neg = False  # 신규 음수 여부
            bonded_new_neg_min = 0.0  # 신규 최저 재고
            bonded_purchase_old_date = None  # 보세에서 함께 당긴 구매전표의 이동 전 날짜 (이력용)
            bonded_purchase_moved = False  # 보세 구매전표 함께 이동 여부

            # ---------- 1순위: 보세→본사 이동 ----------  # 주석
            # 보세 후보: 정확 문구 또는 '07보세창고'+'01본사' 포함 (표기 차이 허용 → BONDED 더 잘 잡히도록)
            party_str = cand[PARTY_COL].astype(str)
            is_bonded = party_str.str.contains(BONDED_MOVE_TEXT, na=False) | (
                party_str.str.contains("07보세창고", na=False) & party_str.str.contains("01본사", na=False)
            )
            cand_bonded = cand[is_bonded].copy()
            if len(cand_bonded) > 0:  # 있으면
                pick_idx = cand_bonded.sort_values(DATE_COL).index[0]  # 1건
                move_type = "BONDED"  # 타입

            # ---------- 2순위: 같은달 매입전표 ----------  # 주석
            if pick_idx is None:  # 보세 없으면
                cand_buy = cand.copy()  # 복사
                cand_buy = cand_buy[~cand_buy[PARTY_COL].astype(str).str.contains(r"\[이동\]", regex=True, na=False)].copy()  # 이동 제외
                cand_buy = cand_buy[(cand_buy[IN_PRICE_COL] > 0.0) & (cand_buy[IN_AMT_COL] > 0.0)].copy()  # 단가/금액
                if len(cand_buy) > 0:  # 있으면
                    run_ts = pd.Timestamp(run_start)
                    cand_buy = cand_buy[(pd.to_datetime(cand_buy[DATE_COL]).dt.year == run_ts.year) & (pd.to_datetime(cand_buy[DATE_COL]).dt.month == run_ts.month)].copy()  # 같은 달 (벡터화)
                if len(cand_buy) > 0:  # 남으면
                    pick_idx = cand_buy.sort_values(DATE_COL).index[0]  # 1건
                    move_type = "PURCHASE"  # 타입

            if pick_idx is None:  # 후보 없으면
                fail_reason = "가져올 입고 없음(보세없음+매입없음/조건불충족)"  # 사유

            old_date = ""  # 이동전 날짜
            old_code = ""  # 이동전 코드
            old_name = ""  # 이동전 품목명
            old_party = ""  # 이동전 거래처

            # ✅ 선택이 있으면 값 채우기  # 주석
            if pick_idx is not None:  # 선택됨
                old_date = pd.Timestamp(df_main.loc[pick_idx, DATE_COL])  # 이전 날짜
                old_code = v_or_blank(df_main.loc[pick_idx, ITEM_COL])  # 이전 코드
                old_name = v_or_blank(df_main.loc[pick_idx, NAME_COL])  # 이전 명
                old_party = v_or_blank(df_main.loc[pick_idx, PARTY_COL])  # 이전 거래처

                # ✅ 보세 타입이면 보세도 같이 이동  # 주석
                if move_type == "BONDED":  # 보세면
                    move_qty = float(df_main.loc[pick_idx, IN_QTY_COL])  # 본사 입고수량(이동)

                    before_mask = (df_bonded[ITEM_COL] == code)  # 품목 마스크
                    before_has_neg = bool((df_bonded.loc[before_mask, STOCK_COL] < -EPS_NEG).any()) if bool(before_mask.any()) else False  # 이전 음수
                    before_min = float(df_bonded.loc[before_mask, STOCK_COL].min()) if bool(before_mask.any()) else 0.0  # 이전 최저(참고)

                    # 날짜는 일자만 비교 (Timestamp vs 문자열/다른 형식 차이로 누락 방지)
                    bd_dates = pd.to_datetime(df_bonded[DATE_COL], errors="coerce").dt.normalize()
                    old_dt = pd.Timestamp(old_date).normalize()
                    mask_bd = (df_bonded[ITEM_COL] == code) & (bd_dates == old_dt)
                    cand_bd = df_bonded.loc[mask_bd].copy()  # 후보

                    if len(cand_bd) == 0:  # 없으면
                        bonded_fail = True  # 실패
                        fail_reason = "보세 전표 없음(날짜/품목 매칭 실패)"  # 사유
                    else:
                        qty_mask = (np.abs(cand_bd[OUT_QTY_COL].astype(np.float64) - move_qty) <= 1e-9)  # 수량 일치 (벡터화)
                        cand_bd2 = cand_bd.loc[qty_mask].copy()  # 필터

                        if len(cand_bd2) == 0:  # 없으면
                            bonded_fail = True  # 실패
                            fail_reason = "보세 전표 수량 불일치(날짜/품목은 일치)"  # 사유
                        else:
                            bidx = cand_bd2.index[0]  # 1건 선택 (보세 출고=창고이동 행)
                            wh_b = df_bonded.loc[bidx, WH_COL]
                            code_b = df_bonded.loc[bidx, ITEM_COL]

                            # ✅ 이동전표 당길 때 보세에서 "그 출고를 채운 구매"도 함께 당겨야 보세 신규 음수 방지
                            # 조건: 같은 창고·품목, 입고수량=move_qty, [이동] 제외, 이동전 날짜 이전/당일, 미사용
                            bd_dates = pd.to_datetime(df_bonded[DATE_COL], errors="coerce").dt.normalize()
                            old_dt_n = pd.Timestamp(old_date).normalize()
                            run_start_n = pd.Timestamp(run_start).normalize()
                            window_start = run_start_n - timedelta(days=int(STEP00_MAX_PULL_DAYS))
                            mask_purchase = (
                                (df_bonded[WH_COL] == wh_b) & (df_bonded[ITEM_COL] == code)
                                & (np.abs(df_bonded[IN_QTY_COL].astype(np.float64) - move_qty) <= 1e-9)
                                & (~df_bonded[PARTY_COL].astype(str).str.contains(r"\[이동\]", regex=True, na=False))
                                & (bd_dates >= window_start) & (bd_dates <= old_dt_n)
                                & (~df_bonded.index.isin(used_bonded_idx))
                            )
                            cand_purchase = df_bonded.loc[mask_purchase].copy()
                            if len(cand_purchase) > 0:
                                cand_purchase = cand_purchase.sort_values(DATE_COL, ascending=False)  # 최근일 우선
                                purchase_idx = cand_purchase.index[0]
                                purchase_old_dt = pd.Timestamp(df_bonded.loc[purchase_idx, DATE_COL])
                                # 달이 바뀌면 구매 이동 금지 (이동후 날짜 run_start와 같은 달일 때만 허용)
                                same_month = (purchase_old_dt.year == run_start.year and purchase_old_dt.month == run_start.month)
                                if same_month:
                                    bonded_purchase_old_date = purchase_old_dt
                                    bonded_purchase_moved = True
                                    df_bonded.loc[purchase_idx, DATE_COL] = run_start  # 구매도 run_start로
                                    df_bonded.loc[purchase_idx, "_moved_first"] = 0  # 해당일 맨 앞(입고 먼저)
                                    used_bonded_idx.add(int(purchase_idx))
                                    if VERBOSE:
                                        print(f"[STEP00] 보세 구매함께이동 | 품목 {code} | 구매 이전일자={bonded_purchase_old_date.date()} → {run_start.date()} | 수량={move_qty}")
                                else:
                                    bonded_purchase_old_date = None
                                    bonded_purchase_moved = False
                                    bonded_fail = True  # 달 바뀌면 창고이동(출고)도 당기지 않음
                                    fail_reason = "보세 구매 이동 시 달 바뀜(창고이동 당기기 생략)"
                                    if VERBOSE:
                                        print(f"[STEP00] 보세 구매 이동 생략(달 바뀜) | 품목 {code} | 구매 {purchase_old_dt.date()} → {run_start.date()} | 창고이동도 당기지 않음")

                            # 달이 바뀌는 경우 출고도 당기지 않음 (위에서 bonded_fail 설정됨)
                            if not bonded_fail:
                                # 보세 출고(창고이동) 행 당기기 → 같은 날 구매 다음에 오도록 _moved_first=1
                                df_bonded.loc[bidx, DATE_COL] = run_start
                                df_bonded.loc[bidx, "_moved_first"] = 1  # 출고는 구매 다음(같은 날 순서: 입고→출고)
                                used_bonded_idx.add(int(bidx))

                                # 해당 보세 그룹만 재계산 (전체 recalc 대체)
                                g_b = df_bonded[(df_bonded[WH_COL] == wh_b) & (df_bonded[ITEM_COL] == code_b)]
                                g_b_new = recalc_inventory_group(g_b)
                                df_bonded.loc[g_b_new.index, STOCK_COL] = g_b_new[STOCK_COL].values

                                after_mask = (df_bonded[ITEM_COL] == code)  # 품목 마스크
                                after_has_neg = bool((df_bonded.loc[after_mask, STOCK_COL] < -EPS_NEG).any()) if bool(after_mask.any()) else False  # 이후 음수
                                after_min = float(df_bonded.loc[after_mask, STOCK_COL].min()) if bool(after_mask.any()) else 0.0  # 이후 최저

                                if (not before_has_neg) and after_has_neg:  # ✅ 신규 음수 (보세에서 이동 후 재계산 시 음수 발생)
                                    bonded_new_neg = True  # 표시
                                    bonded_new_neg_min = after_min  # 기록
                                    if VERBOSE:
                                        print(f"[STEP00] 보세이동후 신규음수 | 품목 {code} | 보세 최저재고={after_min:.2f}")
                                else:
                                    bonded_new_neg = False  # 해제
                                    bonded_new_neg_min = 0.0  # 초기화

                # ✅ 실패 없으면 메인도 이동  # 주석
                if fail_reason == "":  # 성공이면
                    df_main.loc[pick_idx, DATE_COL] = run_start  # ✅ 메인 날짜 당김
                    df_main.loc[pick_idx, "_moved_first"] = 0  # 해당일 맨 앞 배치(같은 날 출고보다 먼저 반영)
                    used_main_idx.add(int(pick_idx))  # 재사용 금지

            # ✅ 실패면 이력 기록하지 않음(요구사항)  # 주석
            if fail_reason != "":  # 실패면
                failed_run_keys.add((wh, code, run_start.date()))  # run 포기
                if VERBOSE:  # 로그면
                    print(f"[STEP00] FAIL| {wh} | {code} | run={run_start.date()} | {fail_reason} | type={move_type} | worst={run['worst']}")  # 출력
                continue  # 다음 run

            # ✅ V18 방식으로 복귀: 이동 1건 성공 시 메인/보세 전체 재계산
            df_main = recalc_inventory(df_main)  # 메인 전체 재계산
            df_bonded = recalc_inventory(df_bonded)  # 보세 전체 재계산

            new_date = pd.Timestamp(df_main.loc[pick_idx, DATE_COL])  # 이동후 날짜
            new_code = v_or_blank(df_main.loc[pick_idx, ITEM_COL])  # 이동후 코드
            new_name = v_or_blank(df_main.loc[pick_idx, NAME_COL])  # 이동후 명
            new_party = v_or_blank(df_main.loc[pick_idx, PARTY_COL])  # 이동후 거래처

            history_rows.append({  # ✅ 성공 이력만 추가
                "이동전 날짜": old_date.date() if isinstance(old_date, pd.Timestamp) else "",  # 이전 날짜
                "이동전 품목코드": old_code,  # 이전 코드
                "이동전 품목명": old_name,  # 이전 명
                "이동전 거래처명": old_party,  # 이전 거래처
                "해당run의 최대 음수": float(run["worst"]),  # run 최저
                "이동후 날짜": run_start.date() if isinstance(run_start, pd.Timestamp) else "",  # 해당 run_start
                "이동후 품목코드": new_code,  # 이후 코드
                "이동후 품목명": new_name,  # 이후 명
                "이동후 거래처명": new_party,  # 이후 거래처
                "이동타입": move_type,  # 타입
                "run_start": run_start.date(),  # run 시작일
                "창고명": v_or_blank(wh),  # 창고명
                "보세이동후_신규음수여부": True if bonded_new_neg else False,  # 신규 음수
                "보세이동후_최저재고": float(bonded_new_neg_min),  # 최저 재고
                "보세_함께이동한_구매_여부": bonded_purchase_moved,  # 보세 구매전표 함께 당김 여부
                "보세_함께이동한_구매_이전일자": bonded_purchase_old_date.date() if bonded_purchase_old_date is not None else None,  # 어떤 구매를 옮겼는지
                "보세_함께이동한_구매_이동후날짜": run_start.date() if bonded_purchase_moved else None,  # 구매가 이동된 날짜(확인용)
            })

            success_count += 1  # 성공 카운트

            if VERBOSE:  # 로그면
                print(f"[STEP00] OK  | {wh} | {code} | {old_date.date()} → {run_start.date()} | type={move_type} | worst={run['worst']}")  # 출력

            if STEP00_SAVE_EVERY_N_SUCCESS > 0 and (success_count % STEP00_SAVE_EVERY_N_SUCCESS == 0):  # 주기 저장
                ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # 시간 문자열
                out_path = f"{STEP00_SAVE_PREFIX}_{ts}.xlsx"  # 파일명
                df_history_mid = pd.DataFrame(history_rows)  # 이력 DF

                # ⚠️ df_main/df_bonded 전체를 중간저장하면 엑셀 쓰기 때문에 매우 느려서 멈춘 것처럼 보일 수 있음
                # 기본은 이력만 저장(가볍게) / 필요할 때만 STEP00_MID_SAVE_FULL=True로 전체 저장
                with pd.ExcelWriter(out_path, engine="openpyxl") as w:  # 저장
                    if STEP00_MID_SAVE_FULL:
                        df_main.to_excel(w, index=False, sheet_name="수불부_중간")  # 메인
                        df_bonded.to_excel(w, index=False, sheet_name="보세_중간")  # 보세
                    df_history_mid.to_excel(w, index=False, sheet_name="이력이력_중간")  # 이력(항상)

                print("중간저장 완료:", out_path)  # 로그

    df_history = pd.DataFrame(history_rows)  # 최종 이력 DF
    if "_moved_first" in df_main.columns:
        df_main = df_main.drop(columns=["_moved_first"])
    if "_moved_first" in df_bonded.columns:
        df_bonded = df_bonded.drop(columns=["_moved_first"])
    return df_main, df_bonded, df_history  # 반환

# =========================  # 구분선
# 7-2) STEP00 보세수불부 구매전표 이동 (같은 달 안에서만)
# =========================  # 구분선
def step00_bonded_pull_same_month(df_bonded):  # 보세수불부 음수 run → 구매전표만 같은 달 안에서 당김
    df_bonded = df_bonded.copy()
    if "_moved_first" not in df_bonded.columns:
        df_bonded["_moved_first"] = 1
    df_bonded = recalc_inventory(df_bonded)
    min_stock = df_bonded.groupby([WH_COL, ITEM_COL])[STOCK_COL].min()
    keys = min_stock[min_stock < -EPS_NEG].index.tolist()
    failed_run_keys = set()
    used_bonded_idx = set()

    iter_keys = tqdm(keys, desc="STEP00(보세)", total=len(keys)) if tqdm else keys  # 진행바
    for wh, code in iter_keys:
        if tqdm:
            iter_keys.set_postfix_str(f"{wh} | {code}")
        while True:
            g = df_bonded[(df_bonded[WH_COL] == wh) & (df_bonded[ITEM_COL] == code)].copy()
            if len(g) == 0 or float(g[STOCK_COL].min()) >= -EPS_NEG:
                break
            g = g.sort_values(DATE_COL).reset_index(drop=False)
            runs = find_all_negative_runs(g)
            if not runs:
                break
            run = None
            for r in runs:
                rk = (wh, code, pd.Timestamp(r["run_start"]).date())
                if rk not in failed_run_keys:
                    run = r
                    break
            if run is None or float(run["worst"]) > float(STEP00_WORST_THRESHOLD):
                break
            run_start = pd.Timestamp(run["run_start"])
            end_day = run_start + pd.offsets.MonthEnd(0)  # 해당 달 말일만 (달 벗어나면 안 됨)
            base_mask = (
                (df_bonded[WH_COL] == wh) & (df_bonded[ITEM_COL] == code)
                & (df_bonded[DATE_COL] > run_start) & (df_bonded[DATE_COL] <= end_day)
                & (df_bonded[IN_QTY_COL] > 0.0)
            )
            cand = df_bonded[base_mask].copy()
            cand = cand[~cand.index.isin(used_bonded_idx)]
            cand = cand[~cand[PARTY_COL].astype(str).str.contains(r"\[이동\]", regex=True, na=False)]
            cand = cand[(cand[IN_PRICE_COL] > 0.0) & (cand[IN_AMT_COL] > 0.0)]
            run_ts = run_start
            cand = cand[(pd.to_datetime(cand[DATE_COL]).dt.year == run_ts.year) & (pd.to_datetime(cand[DATE_COL]).dt.month == run_ts.month)]
            if len(cand) == 0:
                failed_run_keys.add((wh, code, run_start.date()))
                if VERBOSE:
                    print(f"[STEP00_보세] FAIL| {wh} | {code} | run={run_start.date()} | 같은 달 구매전표 없음 | worst={run['worst']}")
                continue
            pick_idx = cand.sort_values(DATE_COL).index[0]
            df_bonded.loc[pick_idx, DATE_COL] = run_start
            df_bonded.loc[pick_idx, "_moved_first"] = 0
            used_bonded_idx.add(int(pick_idx))
            wh_b, code_b = df_bonded.loc[pick_idx, WH_COL], df_bonded.loc[pick_idx, ITEM_COL]
            g_b = df_bonded[(df_bonded[WH_COL] == wh_b) & (df_bonded[ITEM_COL] == code_b)]
            g_b_new = recalc_inventory_group(g_b)
            df_bonded.loc[g_b_new.index, STOCK_COL] = g_b_new[STOCK_COL].values
            if VERBOSE:
                print(f"[STEP00_보세] OK  | {wh} | {code} | → {run_start.date()} (같은 달 구매전표 이동) | worst={run['worst']}")
    if "_moved_first" in df_bonded.columns:
        df_bonded = df_bonded.drop(columns=["_moved_first"])
    return df_bonded

# =========================  # 구분선
# 8) 메인  # 섹션
# =========================  # 구분선
def main():  # 메인
    time_start = time.time()  # 전체 소요 시간 측정 시작
    df_main = preprocess(load_excel(PATH_SUFUL))  # 수불부 로드
    _df_submit = load_excel(PATH_SUBMIT)  # 제출본 로드(미사용)
    df_bonded = preprocess(load_excel(PATH_BONDED))  # 보세 로드

    df_main, df_bonded, df_history = step00_pull_all_wh(df_main, df_bonded)  # STEP00 실행
    df_bonded = step00_bonded_pull_same_month(df_bonded)  # 보세수불부 음수 → 같은 달 구매전표만 이동

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:  # 저장
        df_main.to_excel(w, index=False, sheet_name="수불부_보정본")  # 메인 저장
        df_bonded.to_excel(w, index=False, sheet_name="보세수불부_보정본")  # 보세 저장
        df_history.to_excel(w, index=False, sheet_name="이력이력")  # 이력 저장

    # 전체 소요 시간 출력 (분·초 또는 시·분·초)
    elapsed_sec = time.time() - time_start
    if elapsed_sec < 60:
        time_msg = f"{elapsed_sec:.1f}초"
    elif elapsed_sec < 3600:
        m, s = int(elapsed_sec // 60), elapsed_sec % 60
        time_msg = f"{m}분 {s:.1f}초"
    else:
        h, rest = int(elapsed_sec // 3600), elapsed_sec % 3600
        m, s = int(rest // 60), rest % 60
        time_msg = f"{h}시간 {m}분 {s:.1f}초"
    print("완료:", OUT_XLSX)  # 완료 로그
    print(f"총 소요 시간: {time_msg} ({elapsed_sec:.1f}초)")  # 소요 시간

if __name__ == "__main__":  # 진입점
    main()  # 실행


STEP00(본사):   0%|          | 0/1437 [00:00<?, ?it/s]

[STEP00] 보세 구매 이동 생략(달 바뀜) | 품목 10 | 구매 2025-06-12 → 2025-05-29 | 창고이동도 당기지 않음
[STEP00] FAIL| 01본사창고 | 10 | run=2025-05-29 | 보세 구매 이동 시 달 바뀜(창고이동 당기기 생략) | type=BONDED | worst=-3.0
[STEP00] 보세 구매함께이동 | 품목 10 | 구매 이전일자=2025-07-29 → 2025-07-23 | 수량=200.0
[STEP00] OK  | 01본사창고 | 10 | 2025-07-30 → 2025-07-23 | type=BONDED | worst=-4.0
[STEP00] 보세 구매 이동 생략(달 바뀜) | 품목 10 | 구매 2025-09-01 → 2025-08-20 | 창고이동도 당기지 않음
[STEP00] FAIL| 01본사창고 | 10 | run=2025-08-20 | 보세 구매 이동 시 달 바뀜(창고이동 당기기 생략) | type=BONDED | worst=-3.0
[STEP00] 보세 구매함께이동 | 품목 10 | 구매 이전일자=2025-10-13 → 2025-10-02 | 수량=200.0
[STEP00] OK  | 01본사창고 | 10 | 2025-10-14 → 2025-10-02 | type=BONDED | worst=-1.0
[STEP00] 보세 구매함께이동 | 품목 1002 | 구매 이전일자=2025-05-27 → 2025-05-07 | 수량=300.0
[STEP00] OK  | 01본사창고 | 1002 | 2025-05-28 → 2025-05-07 | type=BONDED | worst=-38.0
[STEP00] 보세 구매 이동 생략(달 바뀜) | 품목 1002 | 구매 2025-12-17 → 2025-11-26 | 창고이동도 당기지 않음
[STEP00] FAIL| 01본사창고 | 1002 | run=2025-11-26 | 보세 구매 이동 시 달 바뀜(창고이동 당기기 생략) | type=BONDED | wor